# Docker 概述

> 容器化概念 / image / container

## 为什么需要它

2013 年以前，部署代码到服务器是一件充满玄学的事。开发者在自己电脑上写完代码一切正常，丢到服务器上报错——Python 版本差一个小版本号、系统缺某个 C 语言库、环境变量路径不一样……排查这些差异往往花掉半天。

当时的解决方案是**虚拟机**（VM）：在物理机上用软件模拟一整套硬件，每个 VM 里装完整操作系统。环境确实一致了，但代价巨大——每个 VM 启动要几分钟、占几个 GB 内存、几 GB 磁盘。一台服务器跑 5 个 VM 就喘不过气。

**Docker**（2013 年发布）用**容器化**给出了新答案：同样环境隔离，但所有容器共享宿主机操作系统内核，启动只需毫秒到秒级，占用 MB 而非 GB。一台机器可以跑几十上百个容器而毫无压力。

## 是什么

Docker 是一个容器化引擎，三个核心概念：

### Image（镜像）—— 只读模板

镜像是一个压缩包，里面有运行一个程序所需的全部东西：代码、语言运行时、系统库、环境变量。它从 **Dockerfile**（一个描述「怎么构建」的文本文件）构建出来，不可修改。

镜像采用了**分层（layer）**设计：每一层只存和上一层的差异。比如「FROM Python + 装依赖 + 拷代码」就是三层，改了代码只重建第三层，前两层复用缓存，构建极快。

### Container（容器）—— 镜像的运行实例

镜像是类，容器是 new 出来的对象。`docker run 镜像名` 就从镜像启动一个容器——一个隔离的进程。同一个镜像可以同时跑几十个容器实例，彼此看不到对方。

### Registry（镜像仓库）—— 存镜像的地方

镜像存在 Registry 里。最大的公共 Registry 是 **Docker Hub**，它之于 Docker 就像 GitHub 之于 Git——别人发布好的镜像（如 `python:3.12`、`nginx`）都可以从上面拉（pull）下来直接用。

### 容器 vs 虚拟机

In [ ]:
虚拟机：                            容器：
  App A  |  App B                     App A  |  App B
  Guest OS | Guest OS       vs.       Docker Engine
  Hypervisor（虚拟化硬件层）             Host OS
  Host OS                             物理硬件
  物理硬件

VM：每个 VM 有独立 OS 内核 + 虚拟硬件，启动分钟级，占 GB 级
容器：所有容器共享 Host OS 内核，启动秒级，占 MB 级


> **Hypervisor**（虚拟机监控程序）：在物理硬件上模拟出多套虚拟硬件（虚拟 CPU、虚拟内存……）的软件层，每个 VM 都以为自己在独占一台物理机。这是 VM 隔离的关键，也是 VM 笨重的原因。

## 怎么样

### 核心操作

In [ ]:
# 从 Docker Hub 拉镜像
docker pull python:3.12

# 查看本地已有的镜像
docker images

# 从镜像启动容器（交互式，进入 bash）
docker run -it python:3.12 bash
# -i 交互模式 -t 分配终端

# 后台运行
docker run -d --name my-app python:3.12 python app.py
# -d 后台 --name 命名容器

# 列出运行中的容器
docker ps

# 列出所有容器（含已停止的）
docker ps -a

# 停止 / 删除
docker stop my-app
docker rm my-app

# 删除镜像
docker rmi python:3.12


### 容器生命周期

```text
pull（下载镜像）→ create（创建容器）→ start → running → stop → rm（删除容器）
                                           ↓
                                         pause（暂停进程，不释放内存）
```

### 端口映射

容器网络默认是隔离的，外部访问不到。用 `-p 宿主机端口:容器端口` 把请求转发进去。

In [ ]:
# 宿主机 8080 → 容器内 nginx 的 80 端口
docker run -p 8080:80 nginx
# 浏览器访问 localhost:8080 即进入容器


### 数据持久化

容器删掉后，里面产生的数据跟着消失。用 `-v`（volume）把宿主机目录「挂」进容器，数据存在宿主机上，不受容器生命周期影响。

In [ ]:
# 把当前目录挂载到容器的 /app
docker run -v $(pwd):/app python:3.12 python /app/script.py


## 相关笔记

- [01-管理工具/05-Docker实战.ipynb](01-管理工具/05-Docker实战.ipynb)——Dockerfile / compose / volume / network
- [09-基础设施/容器底层/容器是什么.ipynb](../09-基础设施/容器底层/容器是什么.ipynb)——namespace / cgroup
- [09-基础设施/Kubernetes/K8s概述.ipynb](../09-基础设施/Kubernetes/K8s概述.ipynb)——容器编排